# P2 diagnostic: training split against validation split (Shakib S-6)

Scores the baseline seeds and the four P2 runs on the training split and the validation split of
v13, with the same evaluation code. Results: `results/v13_trainsplit_eval.json` (unrounded, per
class) and `results/v13_trainsplit_summary.csv`. Resumable: finished jobs are skipped.

**When finished,** restart the kernel. Ultralytics' `val()` can leave dataloader workers alive.


In [ ]:
# === Environment check ===
import os, sys, gc, csv, json, time
from pathlib import Path
import torch, psutil
# Plain tqdm, NOT tqdm.auto: inside a notebook kernel tqdm.auto resolves to the ipywidgets
# frontend, and VS Code then blocks on a "Widgets require us to download supporting files from a
# 3rd party website" dialog (@jupyter-widgets/base). The text bar needs no CDN and no network.
from tqdm import tqdm

print("python     :", sys.version.split()[0])
print("torch      :", torch.__version__, "| cuda", torch.version.cuda)
assert torch.cuda.is_available(), "no CUDA - qml venv needs torch cu130"
print("gpu        :", torch.cuda.get_device_name(0),
      "| cc", torch.cuda.get_device_capability(0),
      "| %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
vm = psutil.virtual_memory()
print("system RAM : %.1f GB total, %.1f GB available (%.0f%% used)"
      % (vm.total/1e9, vm.available/1e9, vm.percent))


In [ ]:
# === Evaluate on the training and validation splits (resumable) ===
import json
import numpy as np, pandas as pd
from ultralytics import YOLO

HERE = Path(r"F:\0BScDefense\04_Code\journal_analysis\v12_local")
os.chdir(HERE)
sys.path.insert(0, str(HERE))
from classes import NAMES, group_of

DATA = Path(r"f:\0BScDefense\05_Datasets\merged-dataset-v13\data.yaml")
RUNS = {"baseline s0": "v13_baseline_s0_640_W2", "baseline s1": "v13_baseline_s1_640_W2",
        "baseline s2": "v13_baseline_s2_640_W2",
        "P2 from scratch": "v13_p2_s0_640_W2", "P2+WIoU from scratch": "v13_p2_wiou_s0_640_W2",
        "P2 pretrained": "v13_p2_rm_s0_640_W2", "P2+WIoU pretrained": "v13_p2_wiou_rm_s0_640_W2"}
SPLITS = ("train", "val")
out_json = HERE / "results" / "v13_trainsplit_eval.json"
done = json.loads(out_json.read_text()) if out_json.exists() else []
have = {(r["run"], r["split"]) for r in done}
jobs = [(n, s) for n in RUNS for s in SPLITS if (n, s) not in have]
print(f"{len(RUNS) * len(SPLITS)} jobs, {len(jobs)} still to run")
bar = tqdm(jobs, unit="job", file=sys.stdout, dynamic_ncols=True)
for name, split in bar:
    bar.set_description(f"{name} | {split}")
    w = HERE / "runs" / RUNS[name] / "weights" / "best.pt"
    b = YOLO(str(w)).val(data=str(DATA), split=split, imgsz=640, workers=0, plots=False,
                         verbose=False, project=str(HERE / "runs"), name="eval_tmp",
                         exist_ok=True).box
    per = {NAMES[int(c)]: {"precision": float(b.p[i]), "recall": float(b.r[i]),
                           "ap50": float(b.ap50[i]), "ap50_95": float(b.ap[i])}
           for i, c in enumerate(b.ap_class_index)}
    done.append({"run": name, "split": split, "map50": float(b.map50), "map50_95": float(b.map),
                 "precision": float(b.mp), "recall": float(b.mr), "per_class": per})
    out_json.write_text(json.dumps(done, indent=1))
    gc.collect(); torch.cuda.empty_cache()
bar.close()


In [ ]:
# === Summary by group ===
rows = []
for r in done:
    per = pd.DataFrame(r["per_class"]).T
    per["group"] = [group_of(c) for c in per.index]
    g = per.groupby("group")[["ap50", "precision", "recall"]].mean()
    rows.append({"run": r["run"], "split": r["split"], "class_mean_ap50": per["ap50"].mean(),
                 "A_ap50": g.loc["A_whole_leaf", "ap50"], "B_ap50": g.loc["B_lesion", "ap50"],
                 "B_precision": g.loc["B_lesion", "precision"], "B_recall": g.loc["B_lesion", "recall"],
                 "map50_95": r["map50_95"]})
S = pd.DataFrame(rows)
S.to_csv(HERE / "results" / "v13_trainsplit_summary.csv", index=False)
wide = S.pivot(index="run", columns="split", values=["class_mean_ap50", "B_ap50"]).round(4)
wide[("gap", "class_mean")] = (wide[("class_mean_ap50", "train")] - wide[("class_mean_ap50", "val")]).round(4)
wide[("gap", "B")] = (wide[("B_ap50", "train")] - wide[("B_ap50", "val")]).round(4)
display(wide.reindex(list(RUNS)))
